In [0]:
%sql

show catalogs;

select current_catalog();

In [0]:
%sql
USE CATALOG lakehouse;

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION `db_s3_external_databricks-s3-ingest-806e3`;

In [0]:
%sql
-- Create additional schemas for different data layers
CREATE SCHEMA IF NOT EXISTS lakehouse.raw
MANAGED LOCATION 's3://katolakehouse/raw'
COMMENT 'Bronze layer - raw ingested data';

CREATE SCHEMA IF NOT EXISTS lakehouse.refined
MANAGED LOCATION 's3://katolakehouse/refined'
COMMENT 'Silver layer - cleaned and transformed data';

CREATE SCHEMA IF NOT EXISTS lakehouse.enterprise
MANAGED LOCATION 's3://katolakehouse/enterprise'
COMMENT 'Gold layer - business-ready aggregated data';


In [0]:
%sql

DESCRIBE SCHEMA lakehouse.enterprise;

In [0]:
%sql

select * from samples.accuweather.forecast_daynight_metric
limit 10;

In [0]:
def load_sample_data(source_table, target_table, description=""):
    """
    Load sample data from Databricks samples to lakehouse.raw schema
    """
    try:
        print(f"Loading {description}...")
        
        # Read from source
        df = spark.table(source_table)
        row_count = df.count()
        
        # Write to target with Delta format
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .option("path", f"s3:///bronze/{target_table.split('.')[-1]}") \
          .saveAsTable(target_table)
        
        print(f"✅ Successfully loaded {row_count} rows to {target_table}")
        return True
        
    except Exception as e:
        print(f"❌ Failed to load {target_table}: {str(e)}")
        return False